In [ ]:
# 04_interaction_and_robustness.ipynb
# Robustness and identification analyses: (i) whether the first-axis disparity is a
# statistical interaction or a cumulative accumulation of marginal effects, across
# all eight attribute combinations and under four model classes; (ii) the CDR
# sensitivity to group-definition cutoffs; (iii) the action-horizon sensitivity; and
# (iv) the search-budget robustness. Tables are written to results/tables. The two
# DiCE-based checks re-use the cached model; set RUN_DICE = True to regenerate them.
# No identifying paths are printed.

from pathlib import Path
import numpy as np, pandas as pd, pickle, warnings, logging
from scipy.stats import beta
warnings.filterwarnings("ignore")

ROOT = Path.cwd(); ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
CACHE = ROOT / "results" / "cache"; TAB = ROOT / "results" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42; TAU = 0.20; RUN_DICE = False

A = pickle.load(open(CACHE / "nb1_artifacts.pkl", "rb"))
rf = A["rf"]; FEAT = A["FEATURES"]; IMMUT = A["IMMUTABLE"]; ACT = A["ACTIONABLE"]; NON = A["NON_CONTROLLABLE"]
Xtr = A["X_train"]; ytr = A["y_train"]; Xte = A["X_test"].copy(); cut = A["cutoffs"]; TARGET = A["TARGET"]
Xte["p"] = rf.predict_proba(Xte[FEAT])[:, 1]

def cmask(df, g, ab, tn):
    gm = df["code_gender"] == (0 if g == "M" else 1)
    am = (df["age"] == 1) if ab == "young" else df["age"].isin([4, 5])
    tm = (df["mn_credit_tenure"] >= cut["long"]) if tn == "long" else (df["mn_credit_tenure"] <= cut["short"])
    return gm & am & tm

# Table 4: eight-cell approval and effect-coded factorial decomposition (log-odds).
cells = []; code = {"M": 1, "F": -1, "young": 1, "old": -1, "long": 1, "short": -1}
X = []; yl = []
for g in ["M", "F"]:
    for ab in ["young", "old"]:
        for tn in ["long", "short"]:
            mean = Xte.loc[cmask(Xte, g, ab, tn), "p"].mean()
            cells.append({"gender": g, "age": ab, "tenure": tn, "P_approval_pct": round(100 * mean, 2)})
            X.append([1, code[g], code[ab], code[tn], code[g]*code[ab], code[g]*code[tn], code[ab]*code[tn], code[g]*code[ab]*code[tn]])
            yl.append(np.log(mean / (1 - mean)))
pd.DataFrame(cells).to_csv(TAB / "table4_interaction_cells.csv", index=False)
bl = np.linalg.solve(np.array(X), np.array(yl))
names = ["intercept", "gender", "age", "tenure", "g_a", "g_t", "a_t", "g_a_t"]
pd.DataFrame([{"term": n, "logit_coef": round(c, 4)} for n, c in zip(names, bl)]).to_csv(TAB / "table4b_interaction_decomposition.csv", index=False)

# Table 6: CDR sensitivity across nine cutoff configurations.
def make_groups(df, old_codes, tenure_short, tenure_long):
    adv = (df["age"] == 1) & (df["code_gender"] == 0) & (df["mn_credit_tenure"] >= tenure_long)
    dis = df["age"].isin(old_codes) & (df["code_gender"] == 1) & (df["mn_credit_tenure"] <= tenure_short)
    g = pd.Series("other", index=df.index); g[adv] = "advantaged"; g[dis] = "disadvantaged"; return g
tq = Xte["mn_credit_tenure"]
TEN = {"p25/p75": (tq.quantile(.25), tq.quantile(.75)), "p20/p80": (tq.quantile(.20), tq.quantile(.80)), "p33/p67": (tq.quantile(.33), tq.quantile(.67))}
AGE = {">=60": [4, 5], ">=50": [3, 4, 5], ">=70": [5]}
rows = []
for tn, (ts, tl) in TEN.items():
    for an, oc in AGE.items():
        g = make_groups(Xte, oc, ts, tl)
        pa = Xte.loc[g == "advantaged", "p"].mean(); pd_ = Xte.loc[g == "disadvantaged", "p"].mean()
        rows.append({"tenure": tn, "age": an, "n_adv": int((g == "advantaged").sum()), "n_dis": int((g == "disadvantaged").sum()),
            "P_adv_pct": round(100*pa, 2), "P_dis_pct": round(100*pd_, 2), "CDR": round(pa/pd_, 1)})
pd.DataFrame(rows).to_csv(TAB / "table6_cdr_sensitivity.csv", index=False)

# Table 7: first-axis disparity across model classes.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
def decomp(proba):
    Xt = Xte.copy(); Xt["q"] = proba; Xd = []; yl2 = []
    for g in ["M", "F"]:
        for ab in ["young", "old"]:
            for tn in ["long", "short"]:
                mm = Xt.loc[cmask(Xt, g, ab, tn), "q"].mean()
                Xd.append([1, code[g], code[ab], code[tn], code[g]*code[ab], code[g]*code[tn], code[ab]*code[tn], code[g]*code[ab]*code[tn]]); yl2.append(np.log(mm/(1-mm)))
    b = np.linalg.solve(np.array(Xd), np.array(yl2))
    adv = Xt.loc[cmask(Xt, "M", "young", "long"), "q"].mean(); dd = Xt.loc[cmask(Xt, "F", "old", "short"), "q"].mean()
    return adv/dd, b[7], 100/(1+np.exp(-(b[0]-b[1]-b[2]-b[3]))), 100*dd
models = {"Random Forest": rf,
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)),
    "Gradient boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "Hist. gradient boosting": HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE)}
rows = []
for name, mdl in models.items():
    pr = mdl.predict_proba(Xte[FEAT])[:, 1] if name == "Random Forest" else mdl.fit(Xtr[FEAT], ytr).predict_proba(Xte[FEAT])[:, 1]
    cdr, tw, add, act = decomp(pr); rows.append({"model": name, "CDR": round(cdr, 1), "three_way_logit": round(tw, 4),
        "additive_pred_pct": round(add, 2), "actual_pct": round(act, 2)})
pd.DataFrame(rows).to_csv(TAB / "table7_classifier_robustness.csv", index=False)

# Tables 8-9: action-horizon and search-budget robustness (disadvantaged, Scenario C).
# These require DiCE; the seed-fixed values are stored for reproducibility and can be
# regenerated by setting RUN_DICE = True (uses the cached rejected pool).
if RUN_DICE:
    import dice_ml; from dice_ml import Dice
    logging.getLogger("dice_ml").setLevel(logging.ERROR)
    frames = pickle.load(open(CACHE / "nb2_frames.pkl", "rb")); dis = frames["dis_full"]
    recmain = pickle.load(open(CACHE / "recourse_main.pkl", "rb")); samp = recmain["disadvantaged"]["_sample"]
    CONT = ["mn_credit_tenure","amt_credit_r3m","amt_usage_r3m","cnt_merchant_r3m","amt_shop","amt_eat","amt_leisure","amt_credit_limit","age"]
    tr = Xtr.copy(); tr[TARGET] = ytr.values
    exp = Dice(dice_ml.Data(dataframe=tr, continuous_features=CONT, outcome_name=TARGET), dice_ml.Model(model=rf, backend="sklearn"), method="random")
    def rng(row, tau): return {f: sorted([float(row[f])*(1-tau), float(row[f])*(1+tau)]) for f in ACT}
    def reliable(o, c):
        imm = any(not np.isclose(float(o[f]), float(c[f])) for f in IMMUT)
        act = any((float(c[f]) != 0) if float(o[f]) == 0 else abs(float(c[f])-float(o[f]))/abs(float(o[f])) > TAU+1e-9 for f in ACT)
        non = any(not np.isclose(float(o[f]), float(c[f])) for f in NON)
        return not imm and not act and not non
    def run(sample, tau, k):
        rel = 0
        for _, row in sample.iterrows():
            try:
                cf = exp.generate_counterfactuals(row[FEAT].to_frame().T, total_CFs=k, desired_class=1, features_to_vary=ACT, permitted_range=rng(row, tau), verbose=False)
                cfdf = cf.cf_examples_list[0].final_cfs_df
                if cfdf is not None and len(cfdf) and any(reliable(row, c) for _, c in cfdf[FEAT].iterrows()): rel += 1
            except Exception: pass
        return rel
    t8 = [{"tau": t, "n_reliable": run(samp, t, 4), "n": len(samp)} for t in [0.10, 0.20, 0.30, 0.50]]
    for r in t8: r["BR_RR_pct"] = round(100*r["n_reliable"]/r["n"], 2)
    t9 = [{"total_CFs": k, "n_reliable": run(dis, 0.20, k), "n": len(dis)} for k in [4, 20]]
    for r in t9: r["BR_RR_pct"] = round(100*r["n_reliable"]/r["n"], 3); r["upper95_pct"] = round(100*(1.0 if r["n_reliable"]==r["n"] else beta.ppf(.975, r["n_reliable"]+1, r["n"]-r["n_reliable"])), 3)
else:
    t8 = [{"tau":0.10,"n_reliable":0,"n":200,"BR_RR_pct":0.00},{"tau":0.20,"n_reliable":0,"n":200,"BR_RR_pct":0.00},
          {"tau":0.30,"n_reliable":0,"n":200,"BR_RR_pct":0.00},{"tau":0.50,"n_reliable":1,"n":200,"BR_RR_pct":0.50}]
    t9 = [{"total_CFs":4,"n_reliable":1,"n":785,"BR_RR_pct":0.127,"upper95_pct":0.708},
          {"total_CFs":20,"n_reliable":2,"n":785,"BR_RR_pct":0.255,"upper95_pct":0.917}]
pd.DataFrame(t8).to_csv(TAB / "table8_horizon_sensitivity.csv", index=False)
pd.DataFrame(t9).to_csv(TAB / "table9_budget_robustness.csv", index=False)

print("Notebook 04 complete. Interaction, CDR-sensitivity, classifier, horizon, and budget tables written.")
